# ULTRA Ablation

This notebook uses the TSV splits exported by `crime_kg_experiments.ipynb` and trains a compact ULTRA mode (similar to Practical 4) for the query:

`(VictimOffenderRelationship, hasRelationshipType, ?)`

The evaluation ranks only valid `RelationshipType` entities as candidate tails. This makes the metric more meaningful because the model is not rewarded for ranking relationship types above thousands of impossible crime entities.

## Setup

In [1]:
!pip install torch-geometric

In [ ]:
from pathlib import Path
from functools import reduce
from collections import defaultdict, Counter
import random

import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils import data as torch_data
from torch_geometric.data import Data
from torch_geometric.nn.conv import MessagePassing
from torch_geometric.utils import scatter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch:', torch.__version__)
print('device:', device)

DATA_DIR = Path('crime_kg_outputs')
TRAIN_PATH = DATA_DIR / 'ultra_train.tsv'
VALID_PATH = DATA_DIR / 'ultra_valid.tsv'
TEST_PATH = DATA_DIR / 'ultra_test.tsv'

for path in [TRAIN_PATH, VALID_PATH, TEST_PATH]:
    print(path, path.exists())


torch: 2.8.0+cu128
device: cuda
crime_kg_outputs/ultra_train.tsv True
crime_kg_outputs/ultra_valid.tsv True
crime_kg_outputs/ultra_test.tsv True


## Load TSV Splits

The files must already exist. If not you can generate them by running the export section in `crime_kg_experiments.ipynb`.

In [3]:
def read_triples(path):
    return pd.read_csv(path, sep='\t', header=None, names=['head', 'relation', 'tail'])

train_df = read_triples(TRAIN_PATH)
valid_df = read_triples(VALID_PATH)
test_df = read_triples(TEST_PATH)

TARGET_REL = 'hasRelationshipType'
train_target_df = train_df[train_df['relation'] == TARGET_REL].copy()
valid_target_df = valid_df[valid_df['relation'] == TARGET_REL].copy()
test_target_df = test_df[test_df['relation'] == TARGET_REL].copy()

print('train triples:', len(train_df))
print('train target:', len(train_target_df))
print('valid target:', len(valid_target_df))
print('test target:', len(test_target_df))
display(train_df['relation'].value_counts().rename_axis('relation').reset_index(name='count'))


train triples: 17995
train target: 946
valid target: 204
test target: 204


,relation,count
0,hasVictim,2133
1,hasOffender,2133
2,relatedToOffender,2131
3,victimInIncident,2068
4,hasVictimType,2068
5,involvedInOffense,1835
6,offenderInIncident,1791
7,hasOffenseType,1445
8,hasIncident,1445
9,hasRelationshipType,946


## Build Entity/Relation vocabularies and PyG Graph

The fact graph contains the training triples plus inverse edges. Validation/test target edges are held out from the message graph.

In [4]:
all_df = pd.concat([train_df, valid_df, test_df], ignore_index=True)
entities = sorted(set(all_df['head']) | set(all_df['tail']))
relations = sorted(set(all_df['relation']))

entity_to_id = {e: i for i, e in enumerate(entities)}
id_to_entity = {i: e for e, i in entity_to_id.items()}
relation_to_id = {r: i for i, r in enumerate(relations)}
id_to_relation = {i: r for r, i in relation_to_id.items()}

num_direct_rel = len(relations)
target_rel_id = relation_to_id[TARGET_REL]

def df_to_direct_tensors(df):
    h = torch.tensor([entity_to_id[x] for x in df['head']] , dtype=torch.long)
    t = torch.tensor([entity_to_id[x] for x in df['tail']], dtype=torch.long)
    r = torch.tensor([relation_to_id[x] for x in df['relation']], dtype=torch.long)
    return h, t, r

def make_fact_graph(df):
    h, t, r = df_to_direct_tensors(df)
    edge_index = torch.stack([h, t], dim=0)
    edge_type = r
    inv_edge_index = edge_index[[1, 0]]
    inv_edge_type = edge_type + num_direct_rel
    graph = Data(
        edge_index=torch.cat([edge_index, inv_edge_index], dim=1),
        edge_type=torch.cat([edge_type, inv_edge_type], dim=0),
        num_nodes=len(entities),
        num_relations=2 * num_direct_rel,
    )
    return graph

train_graph = make_fact_graph(train_df)
print('entities:', len(entities))
print('direct relations:', num_direct_rel)
print('train graph edges incl. inverse:', train_graph.edge_index.size(1))


entities: 8901
direct relations: 10
train graph edges incl. inverse: 35990


## ULTRA Model Code

We use the implementation from Practical 4, adapted slightly.

In [5]:
def construct_relation_graph(graph):
    edge_index, edge_type = graph.edge_index, graph.edge_type
    num_nodes, num_rels = graph.num_nodes, graph.num_relations
    device = edge_index.device

    Eh = torch.vstack([edge_index[0], edge_type]).T.unique(dim=0)
    Eh = torch.sparse_coo_tensor(Eh.T, torch.ones(Eh.shape[0], device=device), (num_nodes, num_rels))
    Eh = Eh.coalesce()
    Eh_idx = Eh.indices()
    Eh_t = torch.sparse_coo_tensor(torch.stack([Eh_idx[1], Eh_idx[0]], dim=0), Eh.values(), (num_rels, num_nodes))

    Et = torch.vstack([edge_index[1], edge_type]).T.unique(dim=0)
    Et = torch.sparse_coo_tensor(Et.T, torch.ones(Et.shape[0], device=device), (num_nodes, num_rels))
    Et = Et.coalesce()
    Et_idx = Et.indices()
    Et_t = torch.sparse_coo_tensor(torch.stack([Et_idx[1], Et_idx[0]], dim=0), Et.values(), (num_rels, num_nodes))

    Ahh = torch.sparse.mm(Eh_t, Eh).coalesce()
    Att = torch.sparse.mm(Et_t, Et).coalesce()
    Aht = torch.sparse.mm(Eh_t, Et).coalesce()
    Ath = torch.sparse.mm(Et_t, Eh).coalesce()

    hh_edges = Ahh.indices()
    tt_edges = Att.indices()
    ht_edges = Aht.indices()
    th_edges = Ath.indices()

    rel_graph = Data(
        edge_index=torch.cat([hh_edges, tt_edges, ht_edges, th_edges], dim=1),
        edge_type=torch.cat([
            torch.zeros(hh_edges.shape[1], dtype=torch.long, device=device),
            torch.ones(tt_edges.shape[1], dtype=torch.long, device=device),
            torch.full((ht_edges.shape[1],), 2, dtype=torch.long, device=device),
            torch.full((th_edges.shape[1],), 3, dtype=torch.long, device=device),
        ], dim=0),
        num_nodes=num_rels,
        num_relations=4,
    )
    graph.relation_graph = rel_graph
    return graph

train_graph = construct_relation_graph(train_graph).to(device)
print(train_graph)
print(train_graph.relation_graph)


Data(edge_index=[2, 35990], edge_type=[35990], num_nodes=8901, num_relations=20, relation_graph=Data(edge_index=[2, 256], edge_type=[256], num_nodes=20, num_relations=4))
Data(edge_index=[2, 256], edge_type=[256], num_nodes=20, num_relations=4)


/tmp/ipykernel_3414/3385579371.py:18: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  Ahh = torch.sparse.mm(Eh_t, Eh).coalesce()


In [6]:
class GeneralizedRelationalConv(MessagePassing):
    def __init__(self, input_dim, output_dim, num_relation, project_relations=False):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.num_relation = num_relation
        self.project_relations = project_relations
        self.linear = nn.Linear(input_dim * 2, output_dim)
        self.layer_norm = nn.LayerNorm(output_dim)
        self.activation = nn.ReLU()
        if not project_relations:
            self.relation = nn.Embedding(num_relation, input_dim)
        else:
            self.relation = None
            self.relation_projection = nn.Sequential(
                nn.Linear(input_dim, input_dim), nn.ReLU(), nn.Linear(input_dim, input_dim)
            )

    def forward(self, x, batch_size, boundary, edge_index, edge_type):
        if not self.project_relations:
            relation = self.relation.weight.expand(batch_size, -1, -1)
        else:
            relation = self.relation_projection(self.relation)
        return self.propagate(x=x, relation=relation, boundary=boundary, edge_index=edge_index, edge_type=edge_type)

    def message(self, x_j, relation, boundary, edge_type):
        relation_j = relation.index_select(1, edge_type)
        message = x_j * relation_j
        return torch.cat([message, boundary], dim=-2)

    def aggregate(self, input, index, dim_size):
        index = torch.cat([index, torch.arange(dim_size, device=input.device)])
        return scatter(input, index, dim=-2, dim_size=dim_size, reduce='sum')

    def update(self, update, x):
        out = self.linear(torch.cat([x, update], dim=-1))
        out = self.layer_norm(out)
        return self.activation(out)

class RelNBFNet(nn.Module):
    def __init__(self, hidden_dims, num_relation=4):
        super().__init__()
        self.dims = hidden_dims
        self.layers = nn.ModuleList([
            GeneralizedRelationalConv(hidden_dims[i], hidden_dims[i + 1], num_relation, project_relations=False)
            for i in range(len(hidden_dims) - 1)
        ])

    def bellmanford(self, data, query_relations):
        batch_size = len(query_relations)
        boundary_query = torch.ones(batch_size, self.dims[0], device=query_relations.device)
        index = query_relations.unsqueeze(-1).expand_as(boundary_query)
        boundary = torch.zeros(batch_size, data.num_nodes, self.dims[0], device=query_relations.device)
        boundary.scatter_add_(1, index.unsqueeze(1), boundary_query.unsqueeze(1))
        x = boundary
        for layer in self.layers:
            layer_output = layer(x, batch_size, boundary, data.edge_index, data.edge_type)
            if layer_output.shape == x.shape:
                layer_output = layer_output + x
            x = layer_output
        return x

    def forward(self, rel_graph, query_relations):
        return self.bellmanford(rel_graph, query_relations)

class EntityNBFNet(nn.Module):
    def __init__(self, hidden_dims, num_relation=1):
        super().__init__()
        self.dims = hidden_dims
        self.layers = nn.ModuleList([
            GeneralizedRelationalConv(hidden_dims[i], hidden_dims[i + 1], num_relation, project_relations=True)
            for i in range(len(hidden_dims) - 1)
        ])
        feature_dim = hidden_dims[0] + hidden_dims[-1]
        self.mlp = nn.Sequential(nn.Linear(feature_dim, feature_dim), nn.ReLU(), nn.Linear(feature_dim, 1))

    def bellmanford(self, data, query_heads, query_relations, relation_representations):
        batch_size = len(query_relations)
        boundary_query = relation_representations[torch.arange(batch_size, device=query_relations.device), query_relations]
        index = query_heads.unsqueeze(-1).expand_as(boundary_query)
        boundary = torch.zeros(batch_size, data.num_nodes, self.dims[0], device=query_relations.device)
        boundary.scatter_add_(1, index.unsqueeze(1), boundary_query.unsqueeze(1))
        x = boundary
        for layer in self.layers:
            layer.relation = relation_representations
            layer_output = layer(x, batch_size, boundary, data.edge_index, data.edge_type)
            if layer_output.shape == x.shape:
                layer_output = layer_output + x
            x = layer_output
        query_relation_expanded = boundary_query.unsqueeze(1).expand(-1, data.num_nodes, -1)
        return torch.cat([x, query_relation_expanded], dim=-1)

    def forward(self, data, relation_representations, batch):
        h_index, t_index, r_index = batch.unbind(-1)
        h_index, t_index, r_index = self.negative_sample_to_tail(
            h_index, t_index, r_index, num_direct_rel=data.num_relations // 2
        )
        out = self.bellmanford(data, h_index[:, 0], r_index[:, 0], relation_representations)
        index = t_index.unsqueeze(-1).expand(-1, -1, out.shape[-1])
        out = out.gather(1, index)
        return self.mlp(out).squeeze(-1).view(h_index.shape)

    def negative_sample_to_tail(self, h_index, t_index, r_index, num_direct_rel):
        is_t_neg = (h_index == h_index[:, [0]]).all(dim=-1, keepdim=True)
        new_h_index = torch.where(is_t_neg, h_index, t_index)
        new_t_index = torch.where(is_t_neg, t_index, h_index)
        new_r_index = torch.where(is_t_neg, r_index, r_index + num_direct_rel)
        return new_h_index, new_t_index, new_r_index

class Ultra(nn.Module):
    def __init__(self, rel_model_cfg, entity_model_cfg):
        super().__init__()
        self.relation_model = RelNBFNet(**rel_model_cfg)
        self.entity_model = EntityNBFNet(**entity_model_cfg)

    def forward(self, data, batch):
        query_relations = batch[:, 0, 2]
        relation_representations = self.relation_model(data.relation_graph, query_relations)
        return self.entity_model(data, relation_representations, batch)


## Candidate-Restricted Training and Evaluation

For this ablation task, negatives are sampled only from relationship-type tails. Evaluation ranks the correct relationship type among those candidate relationship types.

In [7]:
def triples_to_ids(df):
    h = torch.tensor([entity_to_id[x] for x in df['head']] , dtype=torch.long)
    t = torch.tensor([entity_to_id[x] for x in df['tail']], dtype=torch.long)
    r = torch.tensor([relation_to_id[x] for x in df['relation']], dtype=torch.long)
    return torch.stack([h, t, r], dim=1)

train_target = triples_to_ids(train_target_df)
valid_target = triples_to_ids(valid_target_df)
test_target = triples_to_ids(test_target_df)

candidate_tail_ids = sorted(set(train_target[:, 1].tolist()) | set(valid_target[:, 1].tolist()) | set(test_target[:, 1].tolist()))
candidate_tail_ids = torch.tensor(candidate_tail_ids, dtype=torch.long)
print('candidate relationship tails:', len(candidate_tail_ids))
print([id_to_entity[i.item()].split('/')[-1] for i in candidate_tail_ids[:10]])

all_positive_by_query = defaultdict(set)
for triples_tensor in [train_target, valid_target, test_target]:
    for h, t, r in triples_tensor.tolist():
        all_positive_by_query[(h, r)].add(t)

def make_negative_batch(pos_batch, num_neg=16):
    candidates = candidate_tail_ids.to(pos_batch.device)
    rows = []
    for h, t, r in pos_batch.tolist():
        invalid = all_positive_by_query[(h, r)]
        possible = [x for x in candidates.tolist() if x not in invalid]
        if len(possible) >= num_neg:
            neg = random.sample(possible, num_neg)
        else:
            neg = random.choices(possible, k=num_neg)
        tails = [t] + neg
        rows.append([[h, tail, r] for tail in tails])
    return torch.tensor(rows, dtype=torch.long, device=pos_batch.device)

@torch.no_grad()
def evaluate_candidate_ranking(model, data, target_triples, batch_size=16):
    model.eval()
    target_triples = target_triples.to(device)
    candidates = candidate_tail_ids.to(device)
    rankings = []
    for start in range(0, len(target_triples), batch_size):
        batch = target_triples[start:start + batch_size]
        h, t, r = batch[:, 0], batch[:, 1], batch[:, 2]
        h_expand = h.unsqueeze(1).expand(-1, len(candidates))
        r_expand = r.unsqueeze(1).expand(-1, len(candidates))
        t_expand = candidates.unsqueeze(0).expand(len(batch), -1)
        query_batch = torch.stack([h_expand, t_expand, r_expand], dim=-1)
        scores = model(data, query_batch)

        for i in range(len(batch)):
            true_tail = int(t[i].item())
            true_pos = (candidates == true_tail).nonzero(as_tuple=False).view(-1)
            if len(true_pos) != 1:
                continue
            target_score = scores[i, true_pos[0]]
            mask = torch.ones(len(candidates), dtype=torch.bool, device=device)
            for other_true in all_positive_by_query[(int(h[i].item()), int(r[i].item()))]:
                if other_true != true_tail:
                    mask = mask & (candidates != other_true)
            ranking = ((scores[i] > target_score) & mask).sum().item() + 1
            rankings.append(ranking)

    rankings = torch.tensor(rankings, dtype=torch.float)
    return {
        'MR': rankings.mean().item(),
        'MRR': (1.0 / rankings).mean().item(),
        'Hits@1': (rankings <= 1).float().mean().item(),
        'Hits@3': (rankings <= 3).float().mean().item(),
        'Hits@10': (rankings <= 10).float().mean().item(),
    }

def frequency_baseline_metrics(target_triples):
    train_counts = Counter(train_target[:, 1].tolist())
    candidate_scores = {int(c): train_counts[int(c)] for c in candidate_tail_ids.tolist()}
    rankings = []
    for h, t, r in target_triples.tolist():
        target_score = candidate_scores[int(t)]
        rank = 1 + sum(score > target_score for tail, score in candidate_scores.items())
        rankings.append(rank)
    rankings = torch.tensor(rankings, dtype=torch.float)
    return {
        'MR': rankings.mean().item(),
        'MRR': (1.0 / rankings).mean().item(),
        'Hits@1': (rankings <= 1).float().mean().item(),
        'Hits@3': (rankings <= 3).float().mean().item(),
        'Hits@10': (rankings <= 10).float().mean().item(),
    }

print('Frequency valid:', frequency_baseline_metrics(valid_target))
print('Frequency test:', frequency_baseline_metrics(test_target))


candidate relationship tails: 24
['1', '10', '11', '13', '14', '15', '16', '17', '19', '20']
Frequency valid: {'MR': 4.303921699523926, 'MRR': 0.4787021577358246, 'Hits@1': 0.30882352590560913, 'Hits@3': 0.5245097875595093, 'Hits@10': 0.9313725233078003}
Frequency test: {'MR': 4.303921699523926, 'MRR': 0.4723983705043793, 'Hits@1': 0.30882352590560913, 'Hits@3': 0.5098039507865906, 'Hits@10': 0.9460784196853638}


## Train ULTRA on the Crime KG

We train the ULTRA model for 30 epochs for 3 different seeds.

In [8]:
def train_ultra_seed(seed, epochs=30, hidden_dim=32, num_layers=4, batch_size=32, lr=1e-3, weight_decay=1e-5):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    dims = [hidden_dim] * num_layers
    model = Ultra(
        rel_model_cfg={'hidden_dims': dims},
        entity_model_cfg={'hidden_dims': dims},
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    generator = torch.Generator()
    generator.manual_seed(seed)
    train_loader = torch_data.DataLoader(train_target, batch_size=batch_size, shuffle=True, generator=generator)

    num_neg = min(16, max(1, len(candidate_tail_ids) - 1))
    best_valid_mrr = -1
    best_state = None
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        for pos in train_loader:
            pos = pos.to(device)
            batch = make_negative_batch(pos, num_neg=num_neg)
            labels = torch.zeros(batch.shape[:2], device=device)
            labels[:, 0] = 1.0

            optimizer.zero_grad()
            scores = model(train_graph, batch)
            loss = F.binary_cross_entropy_with_logits(scores, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            losses.append(loss.item())

        if epoch % 5 == 0 or epoch == 1:
            valid_metrics = evaluate_candidate_ranking(model, train_graph, valid_target, batch_size=16)
            row = {'seed': seed, 'epoch': epoch, 'loss': float(np.mean(losses)), **{f'valid_{k}': v for k, v in valid_metrics.items()}}
            history.append(row)
            print(row)
            if valid_metrics['MRR'] > best_valid_mrr:
                best_valid_mrr = valid_metrics['MRR']
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    valid_metrics = evaluate_candidate_ranking(model, train_graph, valid_target, batch_size=16)
    test_metrics = evaluate_candidate_ranking(model, train_graph, test_target, batch_size=16)

    return model, pd.DataFrame(history), valid_metrics, test_metrics


ULTRA_SEEDS = [1, 2, 3]
ULTRA_EPOCHS = 30

all_histories = []
ultra_rows = []
trained_models = {}

for seed in ULTRA_SEEDS:
    print(f'===== ULTRA seed {seed} =====')
    model_seed, hist_seed, valid_seed, test_seed = train_ultra_seed(seed, epochs=ULTRA_EPOCHS)
    all_histories.append(hist_seed)
    trained_models[seed] = model_seed
    ultra_rows.append({'model': 'ULTRA-style NBFNet', 'split': 'valid', 'seed': seed, **valid_seed})
    ultra_rows.append({'model': 'ULTRA-style NBFNet', 'split': 'test', 'seed': seed, **test_seed})

history_df = pd.concat(all_histories, ignore_index=True)
ultra_seed_results = pd.DataFrame(ultra_rows)
display(history_df.tail())
display(ultra_seed_results)



===== ULTRA seed 1 =====
{'seed': 1, 'epoch': 1, 'loss': 0.2116837499042352, 'valid_MR': 4.098039150238037, 'valid_MRR': 0.49226832389831543, 'valid_Hits@1': 0.30882352590560913, 'valid_Hits@3': 0.5882353186607361, 'valid_Hits@10': 0.9264705777168274}
{'seed': 1, 'epoch': 5, 'loss': 0.00018172511336160823, 'valid_MR': 4.81862735748291, 'valid_MRR': 0.4761635959148407, 'valid_Hits@1': 0.30882352590560913, 'valid_Hits@3': 0.5245097875595093, 'valid_Hits@10': 0.9019607901573181}
{'seed': 1, 'epoch': 10, 'loss': 2.4225404710402168e-06, 'valid_MR': 6.377450942993164, 'valid_MRR': 0.46024566888809204, 'valid_Hits@1': 0.30882352590560913, 'valid_Hits@3': 0.5245097875595093, 'valid_Hits@10': 0.686274528503418}
{'seed': 1, 'epoch': 15, 'loss': 1.6128051091376014e-06, 'valid_MR': 9.235294342041016, 'valid_MRR': 0.37958043813705444, 'valid_Hits@1': 0.30882352590560913, 'valid_Hits@3': 0.3186274468898773, 'valid_Hits@10': 0.686274528503418}
{'seed': 1, 'epoch': 20, 'loss': 1.5343024301728292e-06, 

,seed,epoch,loss,valid_MR,valid_MRR,valid_Hits@1,valid_Hits@3,valid_Hits@10
16,3,10,0.000008,15.573529,0.079948,0.009804,0.009804,0.328431
17,3,15,0.000003,15.215686,0.083332,0.009804,0.009804,0.328431
18,3,20,0.000002,15.078431,0.084398,0.009804,0.009804,0.328431
19,3,25,0.000002,14.426471,0.094907,0.009804,0.009804,0.328431
20,3,30,0.000001,12.480392,0.141854,0.024510,0.024510,0.333333


,model,split,seed,MR,MRR,Hits@1,Hits@3,Hits@10
0,ULTRA-style NBFNet,valid,1,4.102941,0.492287,0.308824,0.588235,0.931373
1,ULTRA-style NBFNet,test,1,4.068627,0.484558,0.308824,0.573529,0.941176
2,ULTRA-style NBFNet,valid,2,4.171568,0.489681,0.308824,0.588235,0.931373
3,ULTRA-style NBFNet,test,2,4.147059,0.481255,0.308824,0.573529,0.941176
4,ULTRA-style NBFNet,valid,3,4.122549,0.491535,0.308824,0.588235,0.931373
5,ULTRA-style NBFNet,test,3,4.107843,0.482790,0.308824,0.573529,0.946078


## Test Evaluation

In [9]:
freq_valid = frequency_baseline_metrics(valid_target)
freq_test = frequency_baseline_metrics(test_target)

frequency_results = pd.DataFrame([
    {'model': 'frequency baseline', 'split': 'valid', 'seed': 'deterministic', **freq_valid},
    {'model': 'frequency baseline', 'split': 'test', 'seed': 'deterministic', **freq_test},
])

ultra_summary = (
    ultra_seed_results
    .groupby(['model', 'split'], as_index=False)
    .agg(
        MR_mean=('MR', 'mean'), MR_std=('MR', 'std'),
        MRR_mean=('MRR', 'mean'), MRR_std=('MRR', 'std'),
        Hits1_mean=('Hits@1', 'mean'), Hits1_std=('Hits@1', 'std'),
        Hits3_mean=('Hits@3', 'mean'), Hits3_std=('Hits@3', 'std'),
        Hits10_mean=('Hits@10', 'mean'), Hits10_std=('Hits@10', 'std'),
    )
)

freq_summary = frequency_results.rename(columns={
    'MR': 'MR_mean',
    'MRR': 'MRR_mean',
    'Hits@1': 'Hits1_mean',
    'Hits@3': 'Hits3_mean',
    'Hits@10': 'Hits10_mean',
})
for col in ['MR_std', 'MRR_std', 'Hits1_std', 'Hits3_std', 'Hits10_std']:
    freq_summary[col] = 0.0
freq_summary = freq_summary[[
    'model', 'split', 'MR_mean', 'MR_std', 'MRR_mean', 'MRR_std',
    'Hits1_mean', 'Hits1_std', 'Hits3_mean', 'Hits3_std', 'Hits10_mean', 'Hits10_std'
]]

summary = pd.concat([freq_summary, ultra_summary], ignore_index=True)

display(frequency_results)
display(ultra_seed_results)
display(summary)

frequency_results.to_csv(DATA_DIR / 'ultra_frequency_baseline_results.csv', index=False)
ultra_seed_results.to_csv(DATA_DIR / 'ultra_seed_results.csv', index=False)
summary.to_csv(DATA_DIR / 'ultra_candidate_ranking_summary.csv', index=False)



,model,split,seed,MR,MRR,Hits@1,Hits@3,Hits@10
0,frequency baseline,valid,deterministic,4.303922,0.478702,0.308824,0.524510,0.931373
1,frequency baseline,test,deterministic,4.303922,0.472398,0.308824,0.509804,0.946078


,model,split,seed,MR,MRR,Hits@1,Hits@3,Hits@10
0,ULTRA-style NBFNet,valid,1,4.102941,0.492287,0.308824,0.588235,0.931373
1,ULTRA-style NBFNet,test,1,4.068627,0.484558,0.308824,0.573529,0.941176
2,ULTRA-style NBFNet,valid,2,4.171568,0.489681,0.308824,0.588235,0.931373
3,ULTRA-style NBFNet,test,2,4.147059,0.481255,0.308824,0.573529,0.941176
4,ULTRA-style NBFNet,valid,3,4.122549,0.491535,0.308824,0.588235,0.931373
5,ULTRA-style NBFNet,test,3,4.107843,0.482790,0.308824,0.573529,0.946078


,model,split,MR_mean,MR_std,MRR_mean,MRR_std,Hits1_mean,Hits1_std,Hits3_mean,Hits3_std,Hits10_mean,Hits10_std
0,frequency baseline,valid,4.303922,0.000000,0.478702,0.000000,0.308824,0.0,0.524510,0.0,0.931373,0.00000
1,frequency baseline,test,4.303922,0.000000,0.472398,0.000000,0.308824,0.0,0.509804,0.0,0.946078,0.00000
2,ULTRA-style NBFNet,test,4.107843,0.039216,0.482868,0.001653,0.308824,0.0,0.573529,0.0,0.942810,0.00283
3,ULTRA-style NBFNet,valid,4.132353,0.035348,0.491168,0.001341,0.308824,0.0,0.588235,0.0,0.931373,0.00000
